In [1]:
from pathlib import Path
import sys

project_root=Path.cwd().parents[1]
sys.path.append(str(project_root))

In [11]:
from langchain_groq import ChatGroq

from src.query_rewriter.models import(
    QueryRewriteRequest, ChatMessage
)

from src.query_rewriter.models import (RewrittenQueryResult, StepBackResult, ExpansionResult, ReformulationResult)

from src.query_rewriter.reformulator import Reformulator
from src.query_rewriter.expansion import Expansion
from src.query_rewriter.step_back import StepBack
from dotenv import load_dotenv
import os
load_dotenv()


True

In [3]:
groq_api_key=os.getenv("GROQ_API_KEY")

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [5]:
reformulator=Reformulator(llm=llm)

request = QueryRewriteRequest(
    query="How do I install it?",
    history=[
        ChatMessage(
            role="user",
            content="Tell me about SentenceTransformers."
        ),
        ChatMessage(
            role="assistant",
            content="SentenceTransformers is a library for creating sentence embeddings."
        ),
    ],
)


In [6]:
result=reformulator.rewrite(request)

print(result)

ReformulationResult(original_query='How do I install it?', reformulated_query='How to install SentenceTransformers library')


In [7]:
expansion=Expansion(llm=llm)

expanded_queries=expansion.rewrite(request=request)
print(expanded_queries)

ExpansionResult(original_query='How do I install it?', expanded_queries=['How to install SentenceTransformers', 'Installing SentenceTransformers library', 'SentenceTransformers installation guide', 'Download and install SentenceTransformers', 'SentenceTransformers setup and installation', 'Installing SentenceTransformers using pip', 'SentenceTransformers installation instructions', 'How to set up SentenceTransformers', 'Install SentenceTransformers on Python', 'SentenceTransformers installation tutorial'])


In [8]:
step_back=StepBack(llm=llm)

step_back_query=step_back.rewrite(request=request)
print(step_back_query)

StepBackResult(original_query='How do I install it?', step_back_query='What are the general steps to install a Python library?')


In [16]:
class QueryRewriter:

    def __init__(self, reformulator:Reformulator, expansion:Expansion, stepBack:StepBack ):
        self.reformulator=reformulator
        self.expansion=expansion
        self.step_back=stepBack

    def rewrite(self, request:QueryRewriteRequest)-> RewrittenQueryResult:
        
        reformulator_result=self.reformulator.rewrite(request=request)
        
        new_request=QueryRewriteRequest(query= reformulator_result.reformulated_query, history=request.history)
        expansion_result=self.expansion.rewrite(request=new_request)
        step_back_result=self.step_back.rewrite(request=new_request)

        return RewrittenQueryResult(
            original_query=request.query,
            reformulated_query=reformulator_result.reformulated_query,
            step_back_query=step_back_result.step_back_query,
            expanded_queries=expansion_result.expanded_queries
        )
        
        
